# 02 — UMAP Visualization
**Project:** SARS-CoV-2 Genomic Signatures — Explainable Analysis  
**Author:** Ibrahim Mustafa (Bembo)

In [ ]:
!pip install umap-learn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import umap
from scipy import stats

print('Libraries loaded!')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/your_file.csv', sep='\t')
df['Clade'] = df['Clade'].str.replace('.', '', regex=False).str.strip()

X = df.drop(columns=['ID', 'Clade'])
y = df['Clade']

colors = ['#e6194b','#3cb44b','#4363d8','#f58231','#911eb4','#42d4f4','#f032e6']

In [ ]:
# Sample + Outlier removal
sample_idx = df.sample(n=10000, random_state=42).index
X_sample = X.loc[sample_idx]
y_sample = y.loc[sample_idx]

z_scores = np.abs(stats.zscore(X_sample))
mask_clean = (z_scores < 3).all(axis=1)
X_clean = X_sample[mask_clean]
y_clean = y_sample[mask_clean]

print(f'Clean samples: {X_clean.shape[0]}')

In [ ]:
# UMAP
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.1)
X_umap = reducer.fit_transform(X_clean)

plt.figure(figsize=(10, 7))
for i, clade in enumerate(sorted(y_clean.unique())):
    mask = (y_clean == clade).values
    plt.scatter(X_umap[mask, 0], X_umap[mask, 1],
                label=clade, alpha=0.4, s=10, color=colors[i])

plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.title('UMAP of SARS-CoV-2 Genomic Signatures\n(Later clades GRA/GK/GRY show distinct clustering)')
plt.legend(markerscale=3)
plt.tight_layout()
plt.savefig('umap_plot.png', dpi=150)
plt.show()

print('Key observation: GRA (Omicron), GK, and GRY (Alpha) form distinct clusters')
print('Earlier clades (G, GH, GR, GV) overlap — reflecting evolutionary proximity')